# 05 DQA Scene/Class Profile 5h

This notebook is the first DQA experiment aligned to the intended claim:

- server labeled data is limited to the cloudy/partly-cloudy server split
- clients are unlabeled scene domains: highway, city street, residential
- DQA should help when each client has different useful class evidence
- pseudo-label bbox damage is reduced with the best safe profile from the 02/ET probes

Default runtime target is about 5 hours by reusing an existing scene warmup and running a
shorter federated schedule.  For the later 12h run, change `PHASE1_ROUNDS` and
`PHASE2_ROUNDS` in the settings cell.

## 1. Paths

In [1]:
print("hello")

hello


In [2]:
from __future__ import annotations

import json
import os
import re
import shutil
import socket
import subprocess
import sys
from collections import deque
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_profiled.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_profiled.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"

WORK_ROOT = DQA_ROOT / "efficientteacher_dqa05_scene_class_profile_5h"
STATS_ROOT = DQA_ROOT / "stats_dqa05_scene_class_profile_5h"
RUNNER_LOG = DQA_ROOT / "dqa05_scene_class_profile_5h_runner.out"
TRAIN_LOG = DQA_ROOT / "dqa05_scene_class_profile_5h_train.log"
PID_PATH = DQA_ROOT / "dqa05_scene_class_profile_5h_runner.pid"

SOURCE_WARMUP = (
    DQA_ROOT
    / "efficientteacher_dqa_ver2_scene_12h"
    / "global_checkpoints"
    / "round000_warmup.pt"
)

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)
print("source_warmup:", SOURCE_WARMUP)

repo_root: /app/Object_Detection
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h
stats_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa05_scene_class_profile_5h
python: /root/micromamba/envs/al_yolov8/bin/python
source_warmup: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/global_checkpoints/round000_warmup.pt


## 2. Experiment Settings

In [ ]:
# 5h pilot.  For the later 12h run, use PHASE1_ROUNDS=20 and PHASE2_ROUNDS=35.
WARMUP_EPOCHS = 20
PHASE1_ROUNDS = 8
PHASE2_ROUNDS = 14
DQA_START_PHASE = 2

BATCH_SIZE = 160
WORKERS = 6
REQUESTED_GPUS = 2
# The runner deletes completed-round intermediates.  The earlier 70 GiB
# guard is useful for full clean reproductions, but too strict on this
# shared workspace where only ~14 GiB is currently free.
MIN_FREE_GIB = 8

# Loss profile selected by run_dqa_cwa_fedsto_scene_v2_profiled.py.
# strict_low_bbox was chosen because it reduced pseudo-bbox damage while keeping enough
# active classes for DQA.  objectness_only is available as a later ablation.
SSOD_PROFILE = "strict_low_bbox"
CLIENT_LR0 = 3e-4
SERVER_LR0 = 1e-3

SEED_WARMUP_FROM_SOURCE = True
APPEND_TRAIN_LOG = False
RUN_TRAINING = True
RUN_IN_BACKGROUND = False
STREAM_TRAIN_OUTPUT = True

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(29525)

os.environ["DQA05_SSOD_PROFILE"] = SSOD_PROFILE
os.environ["DQA05_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA05_SERVER_LR0"] = str(SERVER_LR0)

{
    "phase1_rounds": PHASE1_ROUNDS,
    "phase2_rounds": PHASE2_ROUNDS,
    "dqa_start_phase": DQA_START_PHASE,
    "ssod_profile": SSOD_PROFILE,
    "client_lr0": CLIENT_LR0,
    "server_lr0": SERVER_LR0,
    "batch_size": BATCH_SIZE,
    "gpus": GPUS,
    "master_port": MASTER_PORT,
    "workspace": str(WORK_ROOT),
}

{'phase1_rounds': 8,
 'phase2_rounds': 14,
 'dqa_start_phase': 2,
 'ssod_profile': 'strict_low_bbox',
 'client_lr0': 0.0003,
 'server_lr0': 0.001,
 'batch_size': 128,
 'gpus': 2,
 'master_port': 29525,
 'workspace': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h'}

## 3. Build Lists and Seed Warmup

The scene setup creates highway/citystreet/residential clients and scene-wise validation
lists.  The 5h pilot reuses the existing scene warmup so the budget is spent on DQA.

In [4]:
subprocess.run(
    [
        str(PYTHON_BIN),
        str(RUN_SCRIPT),
        "--setup-only",
        "--workspace-root",
        str(WORK_ROOT),
        "--stats-root",
        str(STATS_ROOT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=os.environ.copy(),
)

warmup_dst = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
if SEED_WARMUP_FROM_SOURCE and SOURCE_WARMUP.exists() and not warmup_dst.exists():
    warmup_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE_WARMUP, warmup_dst)
    print("Seeded warmup:", warmup_dst)
elif warmup_dst.exists():
    print("Warmup already present:", warmup_dst)
else:
    print("No warmup seed found; the runner will train warmup from scratch.")

manifest = json.loads((WORK_ROOT / "manifest.json").read_text(encoding="utf-8"))
server = manifest["server"]
clients = manifest["clients"]
eval_splits = manifest["paper_evaluation"]["splits"]

display(pd.DataFrame([server]))
display(pd.DataFrame(clients))
display(pd.DataFrame(eval_splits)[["name", "raw_scene", "images", "boxes"]])

{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "48b5cc25459cc3c30d8198067ccff981678f513b",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "48b5cc25459cc3c30d8198067ccff981678f513b",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/dynamic_quality_aware_

,weather,train_list,val_list,source_val_list,train_images,val_images,source_val_images,validation_target
0,cloudy represented by BDD100K Kaggle weather='...,/app/Object_Detection/dynamic_quality_aware_cl...,/app/Object_Detection/dynamic_quality_aware_cl...,/app/Object_Detection/dynamic_quality_aware_cl...,4881,9864,738,scene_total


,id,weather,scene,list,images
0,0,highway,highway,/app/Object_Detection/dynamic_quality_aware_cl...,5000
1,1,citystreet,city street,/app/Object_Detection/dynamic_quality_aware_cl...,5000
2,2,residential,residential,/app/Object_Detection/dynamic_quality_aware_cl...,5000


,name,raw_scene,images,boxes
0,highway,highway,2499,36377
1,citystreet,city street,6112,127178
2,residential,residential,1253,20855


## 4. Dry Run

In [5]:
dry_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--dry-run",
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]

subprocess.run(dry_cmd, cwd=REPO_ROOT, check=True, env=os.environ.copy())

{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "48b5cc25459cc3c30d8198067ccff981678f513b",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "48b5cc25459cc3c30d8198067ccff981678f513b",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/dynamic_quality_aware_

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_profiled.py', '--dry-run', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h', '--stats-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa05_scene_class_profile_5h', '--warmup-epochs', '20', '--phase1-rounds', '8', '--phase2-rounds', '14', '--dqa-start-phase', '2', '--batch-size', '128', '--workers', '4', '--gpus', '2', '--master-port', '29525', '--min-free-gib', '8', '--classwise-blend', '0.35', '--server-anchor', '1.25', '--localize-bn', '--enable-dqa-guard', '--dqa-drop-ratio-threshold', '0.15', '--dqa-spike-ratio-threshold', '3.0'], returncode=0)

## 5. Start or Resume Training

In [6]:
def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


train_cmd = [
    str(PYTHON_BIN),
    "-u",
    str(RUN_SCRIPT),
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--log-file",
    str(TRAIN_LOG),
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]
if APPEND_TRAIN_LOG:
    train_cmd.append("--append-train-log")
if STREAM_TRAIN_OUTPUT:
    train_cmd.append("--stream-train-output")

current_pid = read_pid(PID_PATH)
state = pid_state(current_pid)
print("existing pid:", current_pid, state)
print(" ".join(train_cmd))

if RUN_TRAINING and state not in {"missing", "zombie"}:
    print("Training already appears to be running.")
elif RUN_TRAINING and RUN_IN_BACKGROUND:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "ab" if APPEND_TRAIN_LOG else "wb"
    with RUNNER_LOG.open(log_mode) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=out,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    print("Started PID:", process.pid)
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
elif RUN_TRAINING:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "a" if APPEND_TRAIN_LOG else "w"
    print("Running in foreground; progress will stream in this cell.")
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
    with RUNNER_LOG.open(log_mode, encoding="utf-8", buffering=1) as runner_log:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        PID_PATH.write_text(str(process.pid), encoding="utf-8")
        print("Started PID:", process.pid)
        runner_log.write(f"Started PID: {process.pid}\n")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            runner_log.write(line)
        return_code = process.wait()
    if PID_PATH.exists() and PID_PATH.read_text(encoding="utf-8").strip() == str(process.pid):
        PID_PATH.unlink()
    if return_code != 0:
        raise RuntimeError(f"Training failed with exit code {return_code}. See {RUNNER_LOG} and {TRAIN_LOG}.")
    print("Training completed.")
else:
    print("RUN_TRAINING=False, command was not launched.")

existing pid: None missing
/root/micromamba/envs/al_yolov8/bin/python -u /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_profiled.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa05_scene_class_profile_5h --warmup-epochs 20 --phase1-rounds 8 --phase2-rounds 14 --dqa-start-phase 2 --batch-size 128 --workers 4 --gpus 2 --master-port 29525 --min-free-gib 8 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa05_scene_class_profile_5h_train.log --classwise-blend 0.35 --server-anchor 1.25 --localize-bn --enable-dqa-guard --dqa-drop-ratio-threshold 0.15 --dqa-spike-ratio-threshold 3.0 --stream-train-output
Running in foreground; progress will stream in this cell.
Runner log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa05_scene_

## 6. Status

In [7]:
def tail_lines(path: Path, lines: int = 30) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True, check=True)
        return result.stdout.splitlines()
    except Exception:
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


history_path = WORK_ROOT / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
pid = read_pid(PID_PATH)

completed_phase1 = sum(1 for row in history if int(row.get("phase", 0)) == 1)
completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
latest_global = Path(history[-1]["global"]) if history else WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"

display(
    pd.DataFrame(
        [
            {
                "pid": pid,
                "pid_state": pid_state(pid),
                "completed_phase1": f"{completed_phase1}/{PHASE1_ROUNDS}",
                "completed_phase2": f"{completed_phase2}/{PHASE2_ROUNDS}",
                "completed_total": f"{len(history)}/{PHASE1_ROUNDS + PHASE2_ROUNDS}",
                "latest_global": str(latest_global),
                "free_gib": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
            }
        ]
    )
)

print("Runner log tail:")
for line in tail_lines(RUNNER_LOG, 35):
    print(line)
print("\nTrain log tail:")
for line in tail_lines(TRAIN_LOG, 35):
    print(line)

,pid,pid_state,completed_phase1,completed_phase2,completed_total,latest_global,free_gib
0,None,missing,8/8,14/14,22/22,/app/Object_Detection/dynamic_quality_aware_cl...,127.76


Runner log tail:
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 6/6 [00:06<00:00,  1.01s/it]
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 6/6 [00:06<00:00,  1.01s/it]
                 all        738      14937      0.638      0.431      0.443      0.257
Optimizer stripped from /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/runs/dqa_phase2_round014_server/weights/last.pt, 93.0MB
Optimizer stripped from /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/runs/dqa_phase2_round014_server/weights/best.pt, 93.0MB

Validating /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/runs/dqa_phase2_round014_server/weights/best.pt...
Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 g

## 7. Scene/Class Evaluation

After the run finishes, set `RUN_EVAL=True`.  This evaluates warmup, final phase 1,
and final phase 2 on highway/citystreet/residential/total, with per-class AP rows.

In [8]:
RUN_EVAL = False
EVAL_SPLITS = "highway,citystreet,residential,total"
EVAL_BATCH_SIZE = 16
EVAL_DEVICE = ""

history = json.loads((WORK_ROOT / "history.json").read_text(encoding="utf-8")) if (WORK_ROOT / "history.json").exists() else []
checkpoints: list[tuple[str, Path]] = []
warmup = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
if warmup.exists():
    checkpoints.append(("warmup", warmup))
phase1 = [row for row in history if int(row.get("phase", 0)) == 1]
phase2 = [row for row in history if int(row.get("phase", 0)) == 2]
if phase1:
    checkpoints.append((f"phase1_round{int(phase1[-1]['round']):03d}", Path(phase1[-1]["global"])))
if phase2:
    checkpoints.append((f"phase2_round{int(phase2[-1]['round']):03d}", Path(phase2[-1]["global"])))

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
    "--no-plots",
    "--verbose",
]
if EVAL_DEVICE:
    eval_cmd.extend(["--device", EVAL_DEVICE])
for label, path in checkpoints:
    eval_cmd.extend(["--checkpoint", f"{label}={path}"])

print("checkpoints:", checkpoints)
print(" ".join(eval_cmd))
if RUN_EVAL and checkpoints:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
elif RUN_EVAL:
    print("No checkpoints found to evaluate.")
else:
    print("RUN_EVAL=False; set True after training finishes.")

checkpoints: [('warmup', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/global_checkpoints/round000_warmup.pt')), ('phase1_round008', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/global_checkpoints/phase1_round008_global.pt')), ('phase2_round014', PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/global_checkpoints/phase2_round014_global.pt'))]
/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint warmup=/app/Object_Detection/dynamic_quality_aware_classwise_aggregati

## 8. Read Evaluation Tables

In [9]:
summary_csv = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
classwise_csv = WORK_ROOT / "validation_reports" / "paper_protocol_classwise_summary.csv"

if summary_csv.exists():
    summary = pd.read_csv(summary_csv)
    display(summary.sort_values(["checkpoint_label", "split"]))
else:
    print("No summary yet:", summary_csv)

if classwise_csv.exists():
    classwise = pd.read_csv(classwise_csv)
    display(classwise.sort_values(["split", "class", "map50_95"], ascending=[True, True, False]).head(80))
    pivot = classwise.pivot_table(
        index=["split", "class"],
        columns="checkpoint_label",
        values="map50_95",
        aggfunc="first",
    )
    display(pivot)
else:
    print("No classwise summary yet:", classwise_csv)

No summary yet: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/validation_reports/paper_protocol_eval_summary.csv
No classwise summary yet: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa05_scene_class_profile_5h/validation_reports/paper_protocol_classwise_summary.csv


## 9. DQA Stats Snapshot

In [10]:
state_path = WORK_ROOT / "dqa_cwa_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    guard = state.get("round_guard", {})
    display(pd.DataFrame(guard.get("history", [])).tail(20))
    alpha = state.get("alpha", {})
    if alpha:
        latest_key = sorted(alpha)[-1]
        alpha_df = pd.DataFrame(alpha[latest_key]).T
        alpha_df.columns = latest_key.split("|")
        alpha_df.insert(0, "class", manifest["classes"])
        display(alpha_df)
else:
    print("No DQA state yet:", state_path)

stats_files = sorted(STATS_ROOT.glob("phase*_round*.json"))
print("stats files:", len(stats_files), "root:", STATS_ROOT)

,active_classes,mean_quality,phase,reason,round,total_count,used_dqa
0,8,0.766022,2,,1,377893.0,True
1,8,0.766022,2,,1,377893.0,True
2,8,0.766022,2,,1,377893.0,True
3,9,0.772464,2,,2,350870.0,True
4,9,0.773720,2,,3,350531.0,True
5,9,0.774374,2,,4,350622.0,True
6,9,0.773994,2,,5,355755.0,True
7,9,0.773593,2,,6,362512.0,True
8,9,0.773177,2,,7,366324.0,True
9,9,0.772111,2,,8,369580.0,True


,class,client:0,client:1,client:2,server
0,person,0.165563,0.198897,0.185539,0.45
1,rider,0.154832,0.208682,0.186486,0.45
2,car,0.179620,0.184119,0.186261,0.45
3,bus,0.181228,0.193606,0.175167,0.45
4,truck,0.183506,0.186340,0.180154,0.45
5,bike,0.146697,0.203854,0.199449,0.45
6,motor,0.119399,0.222034,0.208567,0.45
7,traffic light,0.176751,0.190549,0.182700,0.45
8,traffic sign,0.192244,0.180714,0.177042,0.45
9,train,0.250000,0.250000,0.250000,0.25


stats files: 56 root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa05_scene_class_profile_5h
